In [0]:
# 1. Uninstall the conflicting packages
%pip uninstall -y langchain-google-genai langchain langchain-core

# 2. Install the perfectly aligned 0.2.x family, including the compatible Google package (v1.0.8)
%pip install langchain==0.2.16 langchain-core==0.2.43 langchain-google-genai==1.0.8 databricks-langchain langgraph

# 3. Restart the Python engine
dbutils.library.restartPython()

Found existing installation: langchain-google-genai 1.0.8
Uninstalling langchain-google-genai-1.0.8:
  Successfully uninstalled langchain-google-genai-1.0.8
Found existing installation: langchain 0.2.16
Uninstalling langchain-0.2.16:
  Successfully uninstalled langchain-0.2.16
Found existing installation: langchain-core 0.2.43
Uninstalling langchain-core-0.2.43:
  Successfully uninstalled langchain-core-0.2.43
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.4 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.6
    Not uninstalling langchain-core at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-fd01d80a-965e-4b4b-b732-2e6849faf839
    Can't uninstall 'langchain-core'. No files were found to uninstall.
ERROR: pip's dependency resolver

In [0]:
%sql
    
CREATE OR REPLACE FUNCTION agent_monitoring.murugaveltarun.send_teams_alert(
  alert_title STRING COMMENT 'The heading of the alert.',
  alert_message STRING COMMENT 'The detailed explanation generated by the Agent.',
  webhook_url STRING COMMENT 'The secure Microsoft Teams Webhook URL.'
)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Sends a formatted alert message to a Microsoft Teams channel via a webhook. Called by the AI agent after investigating a pipeline failure.'
AS $$
  import json
  import urllib.request

  payload = {
      "title": alert_title,
      "text": alert_message
  }
  
  data = json.dumps(payload).encode('utf-8')
  req = urllib.request.Request(
      webhook_url, 
      data=data, 
      headers={'Content-Type': 'application/json'}
  )
  
  try:
      with urllib.request.urlopen(req) as response:
          if response.status == 200:
              return f"Success: Alert delivered. Status: {response.status}"
          else:
              return f"Failure: Could not deliver alert. Status: {response.status}"
  except Exception as e:
      return f"System Error: Details: {str(e)}"
$$;

In [0]:
# During Agent Execution
teams_webhook = dbutils.secrets.get(scope="synergech_monitoring", key="teams_webhook_url")

# The Agent is prompted to use this specific variable when invoking the tool

In [0]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.databricks import UCFunctionToolkit
from langgraph.prebuilt import create_react_agent

# 1. Fetch the secrets
teams_webhook = dbutils.secrets.get(scope="synergech_monitoring", key="teams_webhook_url")
# Fetch your Gemini API key (Make sure to add it to your Databricks secrets!)
gemini_api_key = dbutils.secrets.get(scope="synergech_monitoring", key="google_api_key") 

# Alternatively, for quick local testing without Databricks secrets:
# gemini_api_key = "AIzaSyYourFreeGeminiKeyHere..."

# 2. Set up Google Environment Variable
os.environ["GOOGLE_API_KEY"] = gemini_api_key

# 3. Connect to Google Gemini
# "gemini-1.5-flash" is the recommended free/fast model, but you can also use "gemini-1.5-pro"
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0
)

# 4. Bind YOUR Unity Catalog Tool
toolkit = UCFunctionToolkit(
    warehouse_id="1a2b3661fe300d416eebc",  # <-- Make sure your Warehouse ID is correct
    function_names=["agent_monitoring.murugaveltarun.send_teams_alert"]
)
tools = toolkit.get_tools()

# 5. The System Prompt 
system_prompt = f"""
You are the Synergech Pipeline Detective, an elite AI monitoring agent.
Your job is to analyze Databricks pipeline error logs and explain them in simple, non-technical terms.

When you receive an error log:
1. Identify the root cause of the failure.
2. Summarize the impact (e.g., "The Driver Liability Assessment failed to run").
3. Suggest a next step for the data engineering team.
4. ONLY after you have done this analysis, use your tool to send the alert to Microsoft Teams.

IMPORTANT SECURITY INSTRUCTION: 
When you invoke the send_teams_alert tool, you MUST use this exact webhook URL for the webhook_url parameter: {teams_webhook}
"""

# 6. Assemble the Agent using LangGraph
agent_executor = create_react_agent(llm, tools, state_modifier=system_prompt)

print("Agent successfully initialized using Google Gemini!")

/databricks/python_shell/lib/dbruntime/autoreload/discoverability/autoreload_discoverability_hook.py:72: UserWarning: Mixing V1 models and V2 models (or constructs, like `TypeAdapter`) is not supported. Please upgrade `BaseMessage` to V2.
  return orig_warn(*args, **kwargs)


Agent successfully initialized using Google Gemini!


In [0]:
import os
import google.generativeai as genai

# Make sure your API key is fetched
gemini_api_key = dbutils.secrets.get(scope="synergech_monitoring", key="google_api_key") 
genai.configure(api_key=gemini_api_key)

print("Available Gemini Models:")
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

Available Gemini Models:
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-max-preview-04-2026


In [0]:
# A simulated error log from your Driver Liability pipeline
messy_error_log = """
Py4JJavaError: An error occurred while calling o112.save.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 14.0 failed 4 times. 
Caused by: java.lang.NullPointerException: Value at column 'Driver_Age' in table 'raw_telemetry' cannot be null.
Constraint validation failed at synergech.liability.assessment.run(liability_calc.py:45)
"""

# Ask the agent to investigate (Notice the 'messages' format!)
print("Waking up Agent to investigate...")
response = agent_executor.invoke({
    "messages": [("user", f"Our pipeline just crashed. Here is the log: {messy_error_log}")]
})

print("\n--- Final Agent Output ---")
print(response["messages"][-1].content)

Waking up Agent to investigate...

--- Final Agent Output ---
### 🔍 Synergech Pipeline Detective Report

**1. Root Cause:**
The pipeline crashed because it encountered a "Null" (empty) value in the `Driver_Age` column within your `raw_telemetry` table. The system is programmed to reject any data that is missing this specific information, and it triggered a safety stop to prevent inaccurate calculations.

**2. Impact:**
The **Driver Liability Assessment** process has failed to complete. As a result, any downstream reports or dashboards relying on this liability data will not be updated until this is resolved.

**3. Next Step for Data Engineering:**
Please investigate the source of the `raw_telemetry` data to determine why `Driver_Age` is missing for some records. You should either implement a data cleaning step to filter out or fill in these missing ages before the assessment runs, or update the source system to ensure this field is mandatory.

***

**Sending alert to Microsoft Teams...